In [1]:
# Setup: add project root
import sys
import os
sys.path.insert(0, '..')

from src.orchestrator import Orchestrator
from src.standards import METADATA_STANDARDS
from src.context.context_factory import create_context
BASE = os.path.abspath(os.path.join('..'))

import logging

# Setup logging for Jupyter notebook
# Force reconfigure by removing existing handlers
logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Remove existing handlers to avoid duplicates
for handler in logger.handlers[:]:
    logger.removeHandler(handler)

# Add a fresh StreamHandler that outputs to stdout (visible in notebook)
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(name)s - %(message)s'))
logger.addHandler(handler)

print("Imports OK")

Imports OK


In [2]:
from src.orchestrator import run_metadata_extraction
multi_source = {
    'biota': os.path.join(BASE, 'data/biota_dataset/biota.csv'),
    'samples': os.path.join(BASE, 'data/biota_dataset/samples.csv'),
    'species': os.path.join(BASE, 'data/biota_dataset/species.csv'),
}

In [3]:
data_context = create_context(
    source=multi_source,
    name='biota_multi'
)
metadata_standard=METADATA_STANDARDS['spatial_ecological']
orchestrator = Orchestrator(topology_name='fast')
plan = orchestrator.generate_plan(
    context=data_context,
    metadata_standard=metadata_standard
)

2026-02-23 15:28:37,410 - INFO - root - PlanExecutor initialized with topology: fast
2026-02-23 15:28:37,410 - INFO - root -   Players per step: 2
2026-02-23 15:28:37,410 - INFO - root -   Debate rounds: 1
2026-02-23 15:28:37,411 - INFO - root -   Player pool: ['data_analyst', 'schema_expert']
2026-02-23 15:28:37,411 - INFO - root - Orchestrator initialized with topology: fast
2026-02-23 15:28:37,411 - INFO - root - ============================================================
2026-02-23 15:28:37,412 - INFO - root - GENERATING PLAN
2026-02-23 15:28:37,412 - INFO - root - Context: biota_multi
2026-02-23 15:28:37,412 - INFO - root - Context type: csv
2026-02-23 15:28:37,412 - INFO - root - Resources: ['biota', 'samples', 'species']
2026-02-23 15:28:37,413 - INFO - root - Multi-resource: True
2026-02-23 15:28:37,413 - INFO - root - ============================================================
2026-02-23 15:28:37,413 - INFO - root - Auto-added 'metadata_generator' for final metadata generati

In [4]:
# Validate plan dataflow
from src.orchestrator.utils import validate_plan_dataflow

if plan:
    # Convert to dict list for validation
    plan_dicts = plan.to_dict_list()
    
    is_valid, message = validate_plan_dataflow(plan_dicts)
    
    if is_valid:
        print(f"✅ {message}")
    else:
        print(f"❌ {message}")
else:
    print("No plan to validate.")

✅ Plan dataflow is valid.


In [9]:
plan.pretty_print()

Plan Steps:
Step 0: get_context_overview
  Rationale: To get an overview of the entire context including all resources.
  Required Artifacts: {}
  Produced Artifacts: ['context_overview']
Step 1: get_context_schema
  Rationale: To understand the schema of the context, including field names, data types, and relationships.
  Required Artifacts: {}
  Produced Artifacts: ['context_schema']
Step 2: get_relationships
  Rationale: To discover and validate relationships between the resources.
  Required Artifacts: {}
  Produced Artifacts: ['discovered_relationships']
Step 3: metadata_generator
  Rationale: To generate concrete metadata values based on the gathered information.
  Required Artifacts: {'metadata_standard': 'metadata_standard', 'context_overview': 'context_overview', 'context_schema': 'context_schema', 'discovered_relationships': 'discovered_relationships'}
  Produced Artifacts: ['metadata_output']


In [6]:
from src.orchestrator.plan_executor import PlanExecutor
from src.tools.context_tools import register_context

context_key = "ctx_biota_multi"
register_context(context_key, data_context)
metadata_standard = METADATA_STANDARDS["spatial_ecological"]
executor = PlanExecutor(topology_name="fast")
result = executor.execute(
    plan=plan,
    context=data_context,
    context_key=context_key,
    metadata_standard=metadata_standard,
    metadata_standard_name="spatial_ecological"
)

2026-02-23 15:30:13,030 - INFO - root - PlanExecutor initialized with topology: fast
2026-02-23 15:30:13,031 - INFO - root -   Players per step: 2
2026-02-23 15:30:13,031 - INFO - root -   Debate rounds: 1
2026-02-23 15:30:13,031 - INFO - root -   Player pool: ['data_analyst', 'schema_expert']
2026-02-23 15:30:13,031 - INFO - root - Using structured output schema: SpatialEcologicalMetadata
2026-02-23 15:30:13,031 - INFO - root - ============================================================
2026-02-23 15:30:13,032 - INFO - root - STARTING PLAN EXECUTION
2026-02-23 15:30:13,032 - INFO - root - Context: biota_multi
2026-02-23 15:30:13,032 - INFO - root - Type: csv
2026-02-23 15:30:13,033 - INFO - root - Resources: ['biota', 'samples', 'species']
2026-02-23 15:30:13,034 - INFO - root - Steps: 4
2026-02-23 15:30:13,034 - INFO - root - ============================================================
2026-02-23 15:30:13,034 - INFO - root - 
2026-02-23 15:30:13,035 - INFO - root - =================

In [ ]:
from pprint import pprint
pprint(result.final_workspace['metadata_output'])

{'description': 'This dataset contains biological data collected from multiple '
                'samples across various tidal basins and flats in the '
                'Netherlands. It includes abundance and biomass data for '
                'different species, along with metadata about the sampling '
                'locations, dates, and methods. The data spans from 2008 to '
                '2024 and was primarily collected using boats with various '
                'sampling methods such as grid and random sampling.',
 'format': 'CSV',
 'methods': 'Sampling conducted primarily using boats with various methods '
            'such as grid and random sampling. Data collected includes '
            'abundance and biomass of species.',
 'spatial_coverage': 'Various tidal basins and flats in the Netherlands, '
                     'including Marsdiep, Eierlandse Gat, Vlie, Balgzand, and '
                     'others.',
 'spatial_resolution': 'Coordinates provided with precision up to 